# Day 9 | 자동화 파이프라인
## Korean Stock Undervaluation Analysis — Automation Pipeline

**목표:**
- DART 2025 사업보고서 공시 자동 감지
- 현재가 자동 재수집
- 괴리율 자동 갱신 + 스크리닝 재실행
- GitHub 자동 푸시 → Streamlit 자동 갱신

**실행 결과 (2026-03-31):**
```
DART 2025 공시: 133/133개 확인
현재가 수집: 136/136개
시장 중앙값 PER: 19.4 → 29.22 (KOSPI +112% 반영)
저평가 종목: 26개 → 16개 (시장 상승으로 감소)
소요시간: 307초
```


## 0. 환경 세팅

In [ ]:
!pip install -q git+https://github.com/FinanceData/FinanceDataReader.git

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import FinanceDataReader as fdr
import requests
import subprocess
import time
import os
import shutil
from datetime import datetime, timedelta

API_KEY      = "여기에_DART_API_키_입력"
GITHUB_TOKEN = "여기에_GitHub_토큰_입력"
BASE_PATH    = '/content/drive/MyDrive/stock-analysis'
APP_PATH     = f'{BASE_PATH}/streamlit_app'

print("✅ 환경 세팅 완료")


## 1. 파이프라인 함수 정의

In [ ]:
def check_new_dart_reports(api_key, target_year='2025'):
    """DART 사업보고서 공시 여부 확인"""
    url = "https://opendart.fss.or.kr/api/list.json"
    df_target = pd.read_csv(
        f'{BASE_PATH}/data/processed/clean_data_final.csv',
        dtype={'Code': str, 'corp_code': str}
    )
    df_target['corp_code'] = df_target['corp_code'].str.zfill(8)
    disclosed = []

    for _, row in df_target.iterrows():
        params = {
            'crtfc_key': api_key,
            'corp_code': row['corp_code'],
            'bgn_de': '20260101',
            'end_de': datetime.now().strftime('%Y%m%d'),
            'pblntf_ty': 'A',
            'page_count': 5,
        }
        try:
            r = requests.get(url, params=params, timeout=10)
            data = r.json()
            if data.get('status') == '000':
                for item in data.get('list', []):
                    if '사업보고서' in item.get('report_nm', ''):
                        disclosed.append(row['Code'])
                        break
        except:
            pass
        time.sleep(0.1)

    print(f"✅ 2025 사업보고서 공시: {len(disclosed)}/{len(df_target)}개")
    return disclosed


def update_current_prices(df):
    """현재가 재수집"""
    prices = {}
    for _, row in df.iterrows():
        try:
            df_p = fdr.DataReader(row['Code'],
                                  datetime.now() - timedelta(days=5))
            if len(df_p) > 0:
                prices[row['Code']] = df_p['Close'].iloc[-1]
        except:
            pass
        time.sleep(0.03)
    df['price'] = df['Code'].map(prices)
    print(f"✅ 현재가 수집: {len(prices)}개")
    return df


def recalculate_gaps(df):
    """괴리율 재계산 (YoY 조정 포함)"""
    df['PER'] = df['price'] / df['EPS']
    df = df[df['PER'] > 0].copy()

    market_per = df['PER'].median()
    sector_per = df.groupby('Sector')['PER'].median()
    df['sector_median_per'] = df['Sector'].map(sector_per)
    df['gap_market'] = (df['price'] - df['EPS'] * market_per) / (df['EPS'] * market_per)
    df['gap_sector'] = (df['price'] - df['EPS'] * df['sector_median_per']) / (df['EPS'] * df['sector_median_per'])

    def adjust(gap, yoy):
        if pd.isna(gap): return None
        if pd.isna(yoy) or yoy <= 0: return round(gap, 4)
        yoy_d = min(yoy, 100.0) / 100.0
        return round(gap - (gap * yoy_d * 0.5), 4) if gap > 0 else round(gap, 4)

    df['gap_market_v2'] = df.apply(lambda r: adjust(r['gap_market'], r.get('yoy_growth_pct')), axis=1)
    df['gap_sector_v2'] = df.apply(lambda r: adjust(r['gap_sector'],  r.get('yoy_growth_pct')), axis=1)

    def signal(row):
        gm, gs = row['gap_market_v2'], row['gap_sector_v2']
        if pd.isna(gm) or pd.isna(gs): return 'UNKNOWN'
        avg = (gm + gs) / 2
        if avg <= -0.30:   return '강력매수'
        elif avg <= -0.15: return '매수'
        elif avg >= 0.30:  return '강력매도'
        elif avg >= 0.15:  return '매도'
        else:              return '중립'

    df['signal_v2']  = df.apply(signal, axis=1)
    df['updated_at'] = datetime.now().strftime('%Y-%m-%d')
    print(f"✅ 괴리율 재계산: {len(df)}개 | 시장PER: {market_per:.2f}")
    return df


def rerun_screening(df):
    """3중 조건 스크리닝"""
    c1 = (df['gap_market_v2'] < -0.15) & (df['gap_sector_v2'] < -0.15)
    c2 = (df['yoy_growth_pct'] > 0) | (df['yoy_growth_pct'].isna())
    c3 = (df['debt_ratio'] < 200)    | (df['debt_ratio'].isna())
    df_final = df[c1 & c2 & c3].sort_values('gap_market_v2').copy()
    print(f"✅ 스크리닝: {len(df_final)}개")
    return df_final


print("✅ 파이프라인 함수 정의 완료!")


## 2. 전체 파이프라인 실행

In [ ]:
def run_pipeline(api_key, github_token, check_dart=True):
    start = datetime.now()
    print(f"파이프라인 시작: {start.strftime('%Y-%m-%d %H:%M')}")

    df = pd.read_csv(f'{BASE_PATH}/data/output/gap_v2.csv',
                     dtype={'Code': str, 'corp_code': str})
    df['corp_code'] = df['corp_code'].str.zfill(8)

    if check_dart:
        check_new_dart_reports(api_key)

    df       = update_current_prices(df)
    df       = recalculate_gaps(df)
    df_final = rerun_screening(df)

    # 저장
    df.to_csv(f'{BASE_PATH}/data/output/gap_v2.csv',
              index=False, encoding='utf-8-sig')
    df_final.to_csv(f'{BASE_PATH}/data/output/gap_v2_final.csv',
                    index=False, encoding='utf-8-sig')
    shutil.copy(f'{BASE_PATH}/data/output/gap_v2.csv',
                f'{APP_PATH}/data/output/gap_v2.csv')
    shutil.copy(f'{BASE_PATH}/data/output/gap_v2_final.csv',
                f'{APP_PATH}/data/output/gap_v2_final.csv')

    # GitHub 푸시
    os.chdir(BASE_PATH)
    subprocess.run(
        f'git remote set-url origin https://{github_token}@github.com/DongWooYun/stock-analysis.git',
        shell=True
    )
    today = datetime.now().strftime('%Y-%m-%d')
    subprocess.run("git add streamlit_app/data/output/", shell=True)
    subprocess.run(f'git commit -m "Auto update: {today}"', shell=True)
    subprocess.run("git push origin main --force", shell=True)

    elapsed = (datetime.now() - start).seconds
    print(f"완료! 소요: {elapsed}초 | 저평가: {len(df_final)}개")
    return df, df_final

# 실행
df_updated, df_final_updated = run_pipeline(
    api_key=API_KEY,
    github_token=GITHUB_TOKEN,
    check_dart=True
)
